# 05 - Runtime Parameters 튜닝

이 노트북에서는 **Agentic Retrieval**의 런타임 파라미터를 조정하여 검색 품질과 성능을 최적화합니다.

## 📋 학습 내용
1. **Reranker Threshold** - 시맨틱 재순위 점수 임계값
2. **Max Sub Queries** - 최대 하위 쿼리 수
3. **Max Runtime** - 최대 실행 시간
4. **Max Output Size** - 최대 출력 크기
5. 파라미터 조합에 따른 결과 비교

## ⚠️ 사전 요구사항
- `01-setup_knowledge_base.ipynb` 실행 완료

## 1. 환경 설정

In [ ]:
import os
import time
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents import KnowledgeBaseRetrievalClient
from azure.search.documents.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalMediumReasoningEffort,
    KnowledgeRetrievalOutputMode,
    KnowledgeSourceRetrievalParameters,
    KnowledgeRetrievalRequestLimits,
)

# 환경 변수 로드
load_dotenv()

# Azure AI Search 설정
search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_api_key = os.environ["SEARCH_ADMIN_KEY"]
search_credential = AzureKeyCredential(search_api_key)

# 리소스 이름
KNOWLEDGE_BASE_NAME = "products-kb"
KNOWLEDGE_SOURCE_NAME = "products-source"

# 클라이언트 생성
index_client = SearchIndexClient(endpoint=search_endpoint, credential=search_credential)
kb_client = KnowledgeBaseRetrievalClient(
    endpoint=search_endpoint,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential
)

print(f"✅ 연결 완료: {search_endpoint}")
print(f"✅ Knowledge Base: {KNOWLEDGE_BASE_NAME}")

## 2. Runtime Parameters 개요

Agentic Retrieval에서 조정 가능한 주요 파라미터들입니다.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    Agentic Retrieval Runtime Parameters                       ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  📊 Knowledge Source Parameters (소스별 설정)                                 ║
║  ┌────────────────────────────┬─────────────────────────────────────────┐   ║
║  │ reranker_threshold         │ 시맨틱 재순위 점수 임계값 (0.0 - 4.0)    │   ║
║  │ max_sub_queries            │ 최대 하위 쿼리 수 (1 - 20)               │   ║
║  │ always_query_source        │ 항상 이 소스 쿼리 (True/False)           │   ║
║  │ include_references         │ 참조 정보 포함 (True/False)              │   ║
║  │ include_reference_source_data │ 원본 데이터 포함 (True/False)         │   ║
║  └────────────────────────────┴─────────────────────────────────────────┘   ║
║                                                                              ║
║  ⏱️ Request Limits (요청 제한)                                                ║
║  ┌────────────────────────────┬─────────────────────────────────────────┐   ║
║  │ max_runtime_in_seconds     │ 최대 실행 시간 (초)                       │   ║
║  │ max_output_size            │ 최대 출력 크기 (바이트)                   │   ║
║  └────────────────────────────┴─────────────────────────────────────────┘   ║
║                                                                              ║
║  🧠 Reasoning Effort                                                          ║
║  ┌────────────────────────────┬─────────────────────────────────────────┐   ║
║  │ minimal                    │ LLM 없이 단순 하이브리드 검색             │   ║
║  │ low                        │ 기본 쿼리 분해 (1-2개 하위 쿼리)          │   ║
║  │ medium                     │ 심층 쿼리 분해 (다중 하위 쿼리)           │   ║
║  │ high                       │ 최대 추론 (복잡한 분석 질의용)            │   ║
║  └────────────────────────────┴─────────────────────────────────────────┘   ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

## 3. 기본 검색 함수 정의

In [ ]:
def run_retrieval_with_params(
    query: str,
    reasoning_effort: str = "low",
    top: int = 5,
    reranker_threshold: float = None,
    max_sub_queries: int = None,
    max_runtime_seconds: int = None,
    max_output_size: int = None,
    always_query_source: bool = None,
    include_references: bool = True,
    include_reference_source_data: bool = False
) -> dict:
    """
    다양한 파라미터로 Agentic Retrieval을 실행합니다.
    
    Returns:
        response: 검색 결과
        elapsed_time: 실행 시간
    """
    # Reasoning Effort 매핑
    effort_map = {
        "minimal": KnowledgeRetrievalMinimalReasoningEffort(),
        "low": KnowledgeRetrievalLowReasoningEffort(),
        "medium": KnowledgeRetrievalMediumReasoningEffort()
    }
    
    # Knowledge Source Parameters 설정
    source_params = None
    if any([reranker_threshold, max_sub_queries, always_query_source is not None]):
        source_params = [
            KnowledgeSourceRetrievalParameters(
                knowledge_source_name=KNOWLEDGE_SOURCE_NAME,
                reranker_threshold=reranker_threshold,
                max_sub_queries=max_sub_queries,
                always_query_source=always_query_source,
                include_references=include_references,
                include_reference_source_data=include_reference_source_data
            )
        ]
    
    # Request Limits 설정
    request_limits = None
    if max_runtime_seconds or max_output_size:
        request_limits = KnowledgeRetrievalRequestLimits(
            max_runtime_in_seconds=max_runtime_seconds,
            max_output_size=max_output_size
        )
    
    # 요청 생성
    request = KnowledgeBaseRetrievalRequest(
        query=query,
        top=top,
        retrieval_reasoning_effort=effort_map.get(reasoning_effort, effort_map["low"]),
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        knowledge_source_parameters=source_params,
        request_limits=request_limits
    )
    
    # 실행 시간 측정
    start_time = time.time()
    response = kb_client.retrieve(request)
    elapsed_time = time.time() - start_time
    
    return {
        "response": response,
        "elapsed_time": elapsed_time
    }

print("✅ 검색 함수 정의 완료")

In [ ]:
def print_result_summary(result: dict, label: str = ""):
    """검색 결과 요약을 출력합니다."""
    response = result["response"]
    elapsed = result["elapsed_time"]
    
    print(f"\n{'=' * 60}")
    if label:
        print(f"🏷️  {label}")
    print(f"{'=' * 60}")
    
    # 기본 정보
    doc_count = len(response.data) if hasattr(response, 'data') and response.data else 0
    print(f"⏱️  실행 시간: {elapsed:.2f}초")
    print(f"📄 반환 문서 수: {doc_count}")
    
    # 상위 3개 문서
    if hasattr(response, 'data') and response.data:
        print(f"\n📋 상위 결과:")
        for i, doc in enumerate(response.data[:3], 1):
            title = doc.get('title', 'N/A')[:40]
            score = doc.get('@search.score', 'N/A')
            reranker_score = doc.get('@search.reranker_score', 'N/A')
            print(f"   [{i}] {title}")
            if reranker_score != 'N/A':
                print(f"       Reranker Score: {reranker_score:.3f}")
    
    # Activity 정보
    if hasattr(response, 'activity') and response.activity:
        print(f"\n🔍 Activity 수: {len(response.activity)}")
        
    # 토큰 사용량
    if hasattr(response, 'usage') and response.usage:
        total = getattr(response.usage, 'total_tokens', 'N/A')
        print(f"💰 Total Tokens: {total}")

print("✅ 결과 출력 함수 정의 완료")

## 4. Reranker Threshold 실험

**Reranker Threshold**는 시맨틱 재순위(Semantic Ranker) 점수의 최소 임계값입니다.
- 범위: `0.0` ~ `4.0`
- 높을수록: 관련성 높은 결과만 반환 (적은 결과)
- 낮을수록: 더 많은 결과 포함 (노이즈 가능성)

In [ ]:
# 테스트 질의
TEST_QUERY = "아기 옷 중에서 겨울에 따뜻하게 입을 수 있는 제품 추천"

print(f"📝 테스트 질의: {TEST_QUERY}")
print("\n" + "#" * 70)
print("🎯 Reranker Threshold 비교 실험")
print("#" * 70)

In [ ]:
# Threshold 0.0 (기본값 - 모든 결과 포함)
result_t0 = run_retrieval_with_params(
    query=TEST_QUERY,
    reasoning_effort="low",
    reranker_threshold=0.0
)
print_result_summary(result_t0, "Reranker Threshold: 0.0 (필터링 없음)")

In [ ]:
# Threshold 1.0 (낮은 필터링)
result_t1 = run_retrieval_with_params(
    query=TEST_QUERY,
    reasoning_effort="low",
    reranker_threshold=1.0
)
print_result_summary(result_t1, "Reranker Threshold: 1.0 (낮은 필터링)")

In [ ]:
# Threshold 2.0 (중간 필터링)
result_t2 = run_retrieval_with_params(
    query=TEST_QUERY,
    reasoning_effort="low",
    reranker_threshold=2.0
)
print_result_summary(result_t2, "Reranker Threshold: 2.0 (중간 필터링)")

In [ ]:
# Threshold 3.0 (높은 필터링)
result_t3 = run_retrieval_with_params(
    query=TEST_QUERY,
    reasoning_effort="low",
    reranker_threshold=3.0
)
print_result_summary(result_t3, "Reranker Threshold: 3.0 (높은 필터링)")

In [ ]:
# Reranker Threshold 비교 요약
print("\n" + "=" * 70)
print("📊 Reranker Threshold 비교 요약")
print("=" * 70)

thresholds = [
    ("0.0", result_t0),
    ("1.0", result_t1),
    ("2.0", result_t2),
    ("3.0", result_t3)
]

print(f"\n{'Threshold':<12} {'문서 수':<10} {'실행 시간':<12}")
print("-" * 40)
for thresh, result in thresholds:
    doc_count = len(result['response'].data) if hasattr(result['response'], 'data') and result['response'].data else 0
    print(f"{thresh:<12} {doc_count:<10} {result['elapsed_time']:.2f}초")

## 5. Max Sub Queries 실험

**Max Sub Queries**는 LLM이 생성할 수 있는 최대 하위 쿼리 수입니다.
- 범위: `1` ~ `20`
- 높을수록: 더 세분화된 검색 (비용 증가)
- 낮을수록: 단순한 검색 (빠른 응답)

In [ ]:
# 복잡한 질의 (하위 쿼리 분해가 유용한 경우)
COMPLEX_QUERY = "3만원 이하의 블루독베이비 또는 압소바 브랜드 제품 중에서 겨울용이면서 리뷰가 좋은 것"

print(f"📝 복잡한 질의: {COMPLEX_QUERY}")
print("\n" + "#" * 70)
print("🎯 Max Sub Queries 비교 실험")
print("#" * 70)

In [ ]:
# Max Sub Queries: 1 (최소)
result_sq1 = run_retrieval_with_params(
    query=COMPLEX_QUERY,
    reasoning_effort="medium",
    max_sub_queries=1
)
print_result_summary(result_sq1, "Max Sub Queries: 1 (최소 분해)")

In [ ]:
# Max Sub Queries: 5 (기본)
result_sq5 = run_retrieval_with_params(
    query=COMPLEX_QUERY,
    reasoning_effort="medium",
    max_sub_queries=5
)
print_result_summary(result_sq5, "Max Sub Queries: 5 (기본)")

In [ ]:
# Max Sub Queries: 10 (높음)
result_sq10 = run_retrieval_with_params(
    query=COMPLEX_QUERY,
    reasoning_effort="medium",
    max_sub_queries=10
)
print_result_summary(result_sq10, "Max Sub Queries: 10 (상세 분해)")

In [ ]:
# Max Sub Queries 비교 요약
print("\n" + "=" * 70)
print("📊 Max Sub Queries 비교 요약")
print("=" * 70)

sub_queries = [
    ("1", result_sq1),
    ("5", result_sq5),
    ("10", result_sq10)
]

print(f"\n{'Max Sub Queries':<18} {'문서 수':<10} {'Activity 수':<12} {'실행 시간':<12}")
print("-" * 55)
for sq, result in sub_queries:
    doc_count = len(result['response'].data) if hasattr(result['response'], 'data') and result['response'].data else 0
    activity_count = len(result['response'].activity) if hasattr(result['response'], 'activity') and result['response'].activity else 0
    print(f"{sq:<18} {doc_count:<10} {activity_count:<12} {result['elapsed_time']:.2f}초")

## 6. Request Limits 실험

**Request Limits**로 실행 시간과 출력 크기를 제한할 수 있습니다.

In [ ]:
print("\n" + "#" * 70)
print("🎯 Request Limits 실험")
print("#" * 70)

# 기본 (제한 없음)
result_no_limit = run_retrieval_with_params(
    query=TEST_QUERY,
    reasoning_effort="medium"
)
print_result_summary(result_no_limit, "제한 없음 (기본)")

In [ ]:
# 시간 제한 (10초)
result_time_limit = run_retrieval_with_params(
    query=TEST_QUERY,
    reasoning_effort="medium",
    max_runtime_seconds=10
)
print_result_summary(result_time_limit, "Max Runtime: 10초")

In [ ]:
# 출력 크기 제한
result_output_limit = run_retrieval_with_params(
    query=TEST_QUERY,
    reasoning_effort="medium",
    max_output_size=10000  # 10KB
)
print_result_summary(result_output_limit, "Max Output Size: 10KB")

## 7. Always Query Source 실험

**always_query_source**를 `True`로 설정하면 LLM의 쿼리 플래닝에 관계없이 항상 해당 소스를 쿼리합니다.

In [ ]:
print("\n" + "#" * 70)
print("🎯 Always Query Source 비교")
print("#" * 70)

# 일반적인 검색 (LLM이 소스 선택)
result_normal = run_retrieval_with_params(
    query="유아용 장난감",
    reasoning_effort="medium",
    always_query_source=False
)
print_result_summary(result_normal, "always_query_source: False (LLM 선택)")

In [ ]:
# 항상 쿼리
result_always = run_retrieval_with_params(
    query="유아용 장난감",
    reasoning_effort="medium",
    always_query_source=True
)
print_result_summary(result_always, "always_query_source: True (항상 쿼리)")

## 8. 최적 파라미터 조합 찾기

다양한 파라미터 조합을 테스트하여 최적의 설정을 찾습니다.

In [ ]:
# 파라미터 조합 실험
print("\n" + "#" * 70)
print("🎯 파라미터 조합 비교")
print("#" * 70)

combinations = [
    {
        "name": "빠른 응답 (품질 희생)",
        "params": {
            "reasoning_effort": "minimal",
            "reranker_threshold": 0.0,
            "max_sub_queries": 1
        }
    },
    {
        "name": "균형잡힌 설정 (권장)",
        "params": {
            "reasoning_effort": "low",
            "reranker_threshold": 1.0,
            "max_sub_queries": 5
        }
    },
    {
        "name": "높은 품질 (비용 증가)",
        "params": {
            "reasoning_effort": "medium",
            "reranker_threshold": 2.0,
            "max_sub_queries": 10
        }
    },
    {
        "name": "엄격한 필터링",
        "params": {
            "reasoning_effort": "medium",
            "reranker_threshold": 3.0,
            "max_sub_queries": 5
        }
    }
]

results_comparison = []
for combo in combinations:
    result = run_retrieval_with_params(
        query=COMPLEX_QUERY,
        **combo["params"]
    )
    results_comparison.append({
        "name": combo["name"],
        "result": result
    })
    print_result_summary(result, combo["name"])

In [ ]:
# 종합 비교 테이블
print("\n" + "=" * 80)
print("📊 파라미터 조합 종합 비교")
print("=" * 80)

print(f"\n{'설정':<25} {'문서 수':<10} {'Activity':<10} {'시간':<10} {'토큰':<10}")
print("-" * 70)

for item in results_comparison:
    response = item['result']['response']
    doc_count = len(response.data) if hasattr(response, 'data') and response.data else 0
    activity_count = len(response.activity) if hasattr(response, 'activity') and response.activity else 0
    elapsed = item['result']['elapsed_time']
    tokens = getattr(response.usage, 'total_tokens', 'N/A') if hasattr(response, 'usage') and response.usage else 'N/A'
    
    print(f"{item['name']:<25} {doc_count:<10} {activity_count:<10} {elapsed:.2f}초{'':<5} {tokens}")

## 9. 권장 설정 가이드

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                     📋 Runtime Parameters 권장 설정 가이드                     ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  🎯 사용 사례별 권장 설정                                                      ║
║                                                                              ║
║  ┌─────────────────────────────────────────────────────────────────────────┐ ║
║  │ 📱 모바일 앱 / 빠른 응답 필요                                             │ ║
║  │    • reasoning_effort: minimal                                          │ ║
║  │    • reranker_threshold: 0.0                                            │ ║
║  │    • max_runtime_seconds: 5                                             │ ║
║  └─────────────────────────────────────────────────────────────────────────┘ ║
║                                                                              ║
║  ┌─────────────────────────────────────────────────────────────────────────┐ ║
║  │ 🖥️ 웹 애플리케이션 / 일반 RAG                                            │ ║
║  │    • reasoning_effort: low                                              │ ║
║  │    • reranker_threshold: 1.0 ~ 1.5                                      │ ║
║  │    • max_sub_queries: 5                                                 │ ║
║  └─────────────────────────────────────────────────────────────────────────┘ ║
║                                                                              ║
║  ┌─────────────────────────────────────────────────────────────────────────┐ ║
║  │ 🔬 분석 / 연구 도구                                                       │ ║
║  │    • reasoning_effort: medium                                           │ ║
║  │    • reranker_threshold: 2.0 ~ 2.5                                      │ ║
║  │    • max_sub_queries: 10                                                │ ║
║  └─────────────────────────────────────────────────────────────────────────┘ ║
║                                                                              ║
║  ┌─────────────────────────────────────────────────────────────────────────┐ ║
║  │ 🎯 정밀 검색 / 법률/의료 문서                                             │ ║
║  │    • reasoning_effort: medium ~ high                                    │ ║
║  │    • reranker_threshold: 3.0+                                           │ ║
║  │    • always_query_source: True                                          │ ║
║  └─────────────────────────────────────────────────────────────────────────┘ ║
║                                                                              ║
║  ⚠️ 주의사항                                                                  ║
║  • reranker_threshold가 너무 높으면 결과가 없을 수 있음                        ║
║  • max_sub_queries가 높으면 비용과 지연 시간 증가                              ║
║  • 프로덕션 배포 전 충분한 테스트 권장                                         ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

## 10. 정리

이 노트북에서 학습한 내용:

1. **Reranker Threshold**: 시맨틱 점수 기반 결과 필터링
2. **Max Sub Queries**: 쿼리 분해 깊이 조절
3. **Request Limits**: 실행 시간/출력 크기 제한
4. **Always Query Source**: 강제 소스 쿼리
5. **파라미터 조합 최적화**: 사용 사례별 권장 설정

In [ ]:
print("\n" + "=" * 70)
print("✅ Runtime Parameters 튜닝 실습 완료!")
print("=" * 70)
print("""
다음 노트북:
- 06-mcp_integration.ipynb: Foundry Agent Service + MCP 연동
- 07-activity_tracing.ipynb: 쿼리 분석 및 디버깅
""")